# Data-Free Bypass Visualizations (D1, D2, D4)

Paper-oriented figures for DisGUIDE-style data-free bypass attacks (D1/D2) and CEM rejection-oracle (D4) against FlowPure on **CIFAR-10**.

## What this notebook produces

| Attack | Figure |
|--------|--------|
| **D1 Latent-Manifold** | Pretrained DDPM + DisGUIDE query grids (early/mid/late steps), training curve, detector scatter |
| **D2 Procedural-Natural** | Same DisGUIDE panels + optional natural-statistics diagnostics |
| **D4 Rejection-Oracle** | CEM adaptation strips, detector/entropy scatter, acceptance time series, schematic+measured manifold panel |

## Assumptions

- Victim checkpoint: `runs/notebook/training-victim-cifar10-vgg16_bn-nodefense/target_model`
- FlowPure checkpoint: `runs/flow_matching/cifar10_flowpure_pgd/checkpoint_latest.pt`
- D1 uses frozen `google/ddpm-cifar10-32` with a trainable latent steering map inside the DisGUIDE loop
- Notebook budgets are **small** (256–384 queries) for interactive use; increase for publication quality

## Critical note for D4

Many defense notebooks use `audit_only=True` so queries are scored but **never blocked**. D4 requires **`audit_only=False`** so rejected queries are dropped from CEM evolution. D1/D2 use audit-only mode here because we only visualize query appearance.

## Expected runtime

- D2 (procedural): ~5–15 min on GPU
- D1 (pretrained DDPM + DisGUIDE): ~15–45 min on GPU (diffusion decode per step)
- D4 (blocking oracle): similar to D1 + benign calibration pass

Figures are written to `runs/notebook/figures/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "bypass_viz_utils.py").exists():
    UTILS_DIR = NOTEBOOK_DIR
else:
    UTILS_DIR = NOTEBOOK_DIR / "notebooks" / "flow_matching_defenses"

PROJECT_ROOT = UTILS_DIR
while not (PROJECT_ROOT / "pyproject.toml").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
for path in (str(SRC_ROOT), str(UTILS_DIR)):
    if path not in sys.path:
        sys.path.insert(0, path)

from flowguard_notebook_utils import apply_nature_style, find_project_root, to_cifar_image_array
from flowguard.attacks.adaptive import high_frequency_energy, total_variation
from flowguard.attacks.bypass import RejectionOracleAttackRunner
from flowguard.attacks.disguide_bypass import (
    LatentManifoldAttackRunner,
    ProceduralNaturalAttackRunner,
)
from flowguard.experiments.spec import AttackKind

import importlib

import bypass_viz_utils as bvu

importlib.reload(bvu)

PROJECT_ROOT = find_project_root(NOTEBOOK_DIR)
DEVICE = "mps"
apply_nature_style()

CONFIG = bvu.default_notebook_config(PROJECT_ROOT, device=DEVICE)
CONFIG = bvu.BypassNotebookConfig(
    project_root=CONFIG.project_root,
    target_checkpoint_dir=CONFIG.target_checkpoint_dir,
    flow_checkpoint=CONFIG.flow_checkpoint,
    device=DEVICE,
    query_budget=3840,
    population_size=24,
    diffusion_model_id="google/ddpm-cifar10-32",
    diffusion_scheduler="ddim",
    diffusion_steps=15,
    figure_dir=PROJECT_ROOT / "runs" / "notebook" / "figures",
)
RUN_ROOT = PROJECT_ROOT / "runs" / "notebook" / "data_free_bypass_viz"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG.figure_dir.mkdir(parents=True, exist_ok=True)

for label, path in (
    ("Project root", PROJECT_ROOT),
    ("Victim checkpoint", CONFIG.target_checkpoint_dir),
    ("FlowPure checkpoint", CONFIG.flow_checkpoint),
    ("Figure output", CONFIG.figure_dir),
):
    status = "OK" if path.exists() else "MISSING"
    print(f"[{status}] {label}: {path}")

d1_extra_preview = bvu.d1_disguide_extra(CONFIG)
print(
    f"[OK] D1 pretrained generator: {d1_extra_preview['generator_hf_model_id']} "
    f"({d1_extra_preview['diffusion_scheduler']}, "
    f"{d1_extra_preview['diffusion_steps']} decode steps, "
    f"{d1_extra_preview['diffusion_train_steps']} train steps)"
)

if DEVICE == "cpu":
    display(Markdown("**Warning:** CPU mode is very slow for D1 (DDPM decode). Use GPU if available."))
elif DEVICE == "mps":
    display(Markdown("**Note:** D1 downloads and runs `google/ddpm-cifar10-32` on first execution."))

## Shared benign reference images

We reuse a small CIFAR-10 test strip as a visual anchor for all attack panels.

In [ ]:
# Audit-only engine for D1/D2 visualization (scores recorded, queries not blocked).
VIS_ENGINE = bvu.build_query_engine(CONFIG, velocity_threshold=1e9, audit_only=True)
BENIGN_IMAGES = bvu.load_benign_reference_images(count=8, download=True)

fig = bvu.plot_image_row(BENIGN_IMAGES, suptitle="Benign CIFAR-10 reference strip")
plt.show()

## D1 — Latent-Manifold Data-Free Extraction

DisGUIDE loop with frozen `google/ddpm-cifar10-32`, trainable latent steering, surrogate flow penalty, and substitute ensemble training.

In [ ]:
D1_EXTRA = bvu.d1_disguide_extra(CONFIG)
print("D1 attack extra:", {k: D1_EXTRA[k] for k in (
    "generator_hf_model_id", "diffusion_scheduler", "diffusion_steps",
    "diffusion_train_steps", "velocity_regularizer_weight", "lambda_div",
)})

d1_spec = bvu.build_bypass_spec(
    CONFIG,
    name="viz-d1-latent-manifold",
    attack_kind=AttackKind.LATENT_MANIFOLD,
    attack_mode="latent_manifold",
    attack_extra=D1_EXTRA,
    attack_batch_size=16,
    projection_name="none",
)

d1_result, d1_trace = bvu.run_traced_disguide_attack(
    LatentManifoldAttackRunner,
    spec=d1_spec,
    engine=VIS_ENGINE,
    output_dir=RUN_ROOT / "d1_latent_manifold",
    trace_max_images=CONFIG.cem_trace_max_images,
)

print(d1_result.metadata)
fig = bvu.plot_disguide_evolution_grid(
    d1_trace,
    BENIGN_IMAGES,
    save_path=CONFIG.figure_dir / "d1_latent_queries.pdf",
)
plt.show()
fig = bvu.plot_disguide_training_curve(
    d1_trace,
    save_path=CONFIG.figure_dir / "d1_training_curve.pdf",
)
plt.show()
fig = bvu.plot_disguide_detector_entropy_scatter(
    d1_trace,
    save_path=CONFIG.figure_dir / "d1_detector_entropy.pdf",
)
plt.show()

## D2 — Strict Data-Free Natural-Statistics Generator

Procedural CEM over grid / Fourier / blob parameters (no neural decoder).

In [ ]:
d2_extra = {
    **bvu.disguide_bypass_common_extra(CONFIG),
    "procedural_grid_size": 4,
    "procedural_fourier_modes": 3,
    "procedural_blobs": 4,
    "lambda_tv": 0.01,
    "lambda_freq": 0.01,
    "lambda_color": 0.01,
}
d2_spec = bvu.build_bypass_spec(
    CONFIG,
    name="viz-d2-procedural-natural",
    attack_kind=AttackKind.PROCEDURAL_NATURAL,
    attack_mode="procedural_natural",
    attack_extra=d2_extra,
    projection_name="none",
)

d2_result, d2_trace = bvu.run_traced_disguide_attack(
    ProceduralNaturalAttackRunner,
    spec=d2_spec,
    engine=VIS_ENGINE,
    output_dir=RUN_ROOT / "d2_procedural_natural",
    trace_max_images=CONFIG.cem_trace_max_images,
)

print(d2_result.metadata)
fig = bvu.plot_disguide_evolution_grid(
    d2_trace,
    BENIGN_IMAGES,
    save_path=CONFIG.figure_dir / "d2_procedural_queries.pdf",
)
plt.show()
fig = bvu.plot_disguide_training_curve(
    d2_trace,
    save_path=CONFIG.figure_dir / "d2_training_curve.pdf",
)
plt.show()

# Optional natural-statistics diagnostic on the final elite strip.
if d2_trace and d2_trace[-1].query_images:
    sample = torch.as_tensor(d2_trace[-1].query_images[0], dtype=torch.float32).unsqueeze(0)
    print(
        f"Final elite TV={float(total_variation(sample)):.4f}, "
        f"HF={float(high_frequency_energy(sample)):.4f}"
    )

## D1 vs D2 — attack query comparison (iter 1 & 16)

2×2 panel: one representative population query per cell — D1 on top, D2 on bottom; iter 1 left, iter 16 right.

In [ ]:
# 2x2 grid: one attack query per cell (D1 top, D2 bottom; iter 1 left, iter 16 right).
d1_trace = d1_trace or bvu.load_disguide_trace(RUN_ROOT / "d1_latent_manifold" / "disguide_trace.pt")
d2_trace = d2_trace or bvu.load_disguide_trace(RUN_ROOT / "d2_procedural_natural" / "disguide_trace.pt")

fig = bvu.plot_d1_d2_disguide_query_grid(
    d1_trace,
    d2_trace,
    iterations=(1, 121),
    save_path=CONFIG.figure_dir / "d1_d2_attack_query_grid.pdf",
)
plt.show()

## D4 — Detector-Oracle Rejection Attack

FlowPure is calibrated on benign traffic, then run in **blocking mode**. Only accepted queries update CEM.

In [ ]:
# Fresh engine for calibration (audit-only while measuring benign scores).
CAL_ENGINE = bvu.build_query_engine(CONFIG, velocity_threshold=1e9, audit_only=True)
FLOWPURE_THRESHOLD = bvu.calibrate_flowpure_threshold(
    CAL_ENGINE,
    num_queries=CONFIG.benign_calibration_queries,
    target_fpr=CONFIG.target_fpr,
    download=True,
)
print(f"Calibrated FlowPure threshold (target FPR={CONFIG.target_fpr:.2f}): {FLOWPURE_THRESHOLD:.4f}")

BLOCK_ENGINE = bvu.build_query_engine(
    CONFIG,
    velocity_threshold=FLOWPURE_THRESHOLD,
    audit_only=False,
)

d4_spec = bvu.build_bypass_spec(
    CONFIG,
    name="viz-d4-rejection-oracle",
    attack_kind=AttackKind.REJECTION_ORACLE,
    attack_mode="rejection_oracle",
    attack_extra={
        **bvu.cem_common_extra(CONFIG),
        "proposal": "latent",
        "accepted_only": True,
    },
    projection_name="gaussian_blur",
    projection_strength=0.1,
    projection_steps=10,
)

d4_result, d4_trace = bvu.run_traced_attack(
    RejectionOracleAttackRunner,
    spec=d4_spec,
    engine=BLOCK_ENGINE,
    output_dir=RUN_ROOT / "d4_rejection_oracle",
    trace_max_images=CONFIG.cem_trace_max_images,
)

print(d4_result.metadata)

In [ ]:
if not d4_trace:
    raise RuntimeError("D4 trace is empty; check query budget and blocking threshold.")

strip_iterations = [
    d4_trace[0].iteration,
    d4_trace[min(4, len(d4_trace) - 1)].iteration,
    d4_trace[min(9, len(d4_trace) - 1)].iteration,
    d4_trace[-1].iteration,
]
strip_iterations = sorted(set(strip_iterations))

fig = bvu.plot_cem_iteration_strip(
    d4_trace,
    iterations=strip_iterations,
    save_path=CONFIG.figure_dir / "d4_iteration_strip.pdf",
)
plt.show()

fig = bvu.plot_detector_entropy_scatter(
    d4_trace,
    save_path=CONFIG.figure_dir / "d4_detector_entropy_scatter.pdf",
)
plt.show()

fig = bvu.plot_acceptance_timeseries(
    d4_trace,
    save_path=CONFIG.figure_dir / "d4_acceptance_timeseries.pdf",
)
plt.show()

# Final elite grid: only accepted queries that survived rejection.
final_elite = d4_trace[-1].elite_queries
if final_elite:
    fig = bvu.plot_image_row(
        final_elite[:8],
        suptitle="D4 final accepted elite queries",
    )
    fig.savefig(CONFIG.figure_dir / "d4_final_elite.pdf", dpi=300, bbox_inches="tight")
    plt.show()

### Schematic manifold + measured PCA features

The background density is **conceptual** (adapted from `scripts/generate_teaser_image.py`). Scatter points are **measured** PCA projections of `[detector score, entropy, TV, HF energy]`.

In [ ]:
# Score benign features in audit-only mode (never pass BLOCK_ENGINE here).
benign_features = bvu.collect_manifold_benign_features(CONFIG, count=64, download=False)

# Clearer for papers: all groups in one PCA space.
fig = bvu.plot_manifold_pca_combined(
    d4_trace,
    benign_features,
    early_iterations=2,
    samples_per_group=32,
    save_path=CONFIG.figure_dir / "d4_manifold_pca_combined.pdf",
)
plt.show()

display(
    Markdown(
        "Saved figures under "
        f"`{CONFIG.figure_dir}`: "
        "`d1_latent_queries.pdf`, `d2_procedural_queries.pdf`, "
        "`d4_iteration_strip.pdf`, `d4_detector_entropy_scatter.pdf`, "
        "`d4_acceptance_timeseries.pdf`, `d4_final_elite.pdf`, `d4_manifold_teaser.pdf`."
    )
)